# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook provides an example for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Croissant schema URL:** https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
- **Title:** Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency
- **Description:** Structured tabulations of clinical features, hormone levels, adrenal imaging, and genetic variants for three female patients diagnosed with non-classical 21-hydroxylase deficiency-associated infertility.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

In Croissant, a dataset may contain multiple record sets (tables), each identified by a unique `@id`. We list all record sets and their fields, referencing every entity by its `@id`.

In [ ]:
# List all record sets and their field IDs
record_set_ids = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]

print('Record Set IDs:')
for rsid in record_set_ids:
    print(f'- {rsid}')

# For each record set, list field IDs
for rsid in record_set_ids:
    rs_metadata = next((rs for rs in dataset.metadata.to_json().get('recordSet', []) if rs['@id'] == rsid), None)
    if rs_metadata:
        print(f"\nRecord Set {rsid} fields:")
        fields = rs_metadata.get('field', [])
        for field in fields:
            print(f"  - {field['@id']} ({field.get('name', 'unknown name')})")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use record set and field `@id`s.

Below, we demonstrate extracting all available tables into DataFrames, referencing each by their record set `@id`.

In [ ]:
# Extract data from each record set
record_sets = record_set_ids
dataframes = {}

for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)
    print(f"Record set: {record_set} columns:")
    print(dataframes[record_set].columns.tolist())
    print(dataframes[record_set].head(2))

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records, normalizing numeric fields, and grouping data.

Here, we'll select a numeric field for analysis from one of the available record sets, reference all fields by their `@id`, and demonstrate filtering and normalization.

In [ ]:
# Example: Select a numeric field from first record set
if record_sets:
    record_set_id = record_sets[0]
    df = dataframes[record_set_id]

    # Try to automatically select a numeric field based on the column types
    numeric_fields = [col for col in df.columns if (df[col].dtype == 'float64' or df[col].dtype == 'int64')]

    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to automatically select a group field
        group_fields = [col for col in df.columns if (df[col].dtype == 'object') and ('id' in col.lower() or 'type' in col.lower())]
        if group_fields:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found in the selected record set.")
else:
    print("No record sets detected in the dataset metadata.")

## 5. Visualization
Visualize data distributions or relationships between fields.

Below, we plot the distribution of the chosen numeric field for the first available record set.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} in record set {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion
This notebook demonstrates loading, overviewing, extracting, processing, and visualizing the FAIR^2 dataset package using `mlcroissant`.

- We referenced all data entities by their `@id` as per Croissant schema best practice.
- The dataset comprises multiple record sets, each offering clinical, laboratory, and genetic data for three female patients with NC-21OHD and infertility.
- Exploratory analysis and normalization operations were performed as examples.
- For further research, examine subtle genotype–phenotype relationships or cross-reference hormone levels with genetic variants.
